# 07 · Trajectory analysis — stability, orientation, entrance lid, zinc site

**Milestone 7.** Reads the production trajectory that notebook 06 saves on Drive (`outputs/md_run/`). It answers:

- **A–B:** which parts of the protein are moving?
- **C:** how does the protein sit in the membrane (helix tilt, ectodomain tilt)?
- **D:** does the entrance lid open or close?
- **E:** does the binuclear zinc site stay intact?

The model is a hybrid:

| Region (UniProt, each chain) | Source | Expected behaviour |
|---|---|---|
| 1–19 cytoplasmic tail | AlphaFold (low confidence) | very flexible |
| 20–43 transmembrane helix | AlphaFold | moves as a body in the membrane |
| 44–61 stalk | AlphaFold | flexible linker |
| **62–750 ectodomain** | **9S7U crystal structure** | **should stay close to the start** |
| 541–548 entrance lid | crystal | can open and close |

Every frame is superposed on the crystal ectodomain of both chains. The RMSD of each region then shows how it moves **relative to** the well-determined part. "All Cα, global fit" is the number notebook 06 logs.

**Runtime:** a **CPU** runtime is enough: only numpy is used, and no GPU units are needed. It can run while production runs in another session; it only reads finished segments.

In [ ]:
#@title 0 · the pipeline
import os, sys, subprocess
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
DATA_DIR = None
REPO = "https://github.com/victorolatunde23-dev/psma_structural_modelling.git"
if IN_COLAB and not Path("psma_structural_modelling").exists() and not Path("src/psma_pipeline").exists():
    subprocess.run(["git", "clone", "-q", REPO], check=True)
if Path("psma_structural_modelling").exists():
    os.chdir("psma_structural_modelling")
if IN_COLAB:
    subprocess.run(["git", "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
sys.path.insert(0, str(Path("src").resolve()))
for _m in [m for m in list(sys.modules) if m == "psma_pipeline" or m.startswith("psma_pipeline.")]:
    del sys.modules[_m]
import psma_pipeline
print("psma_pipeline", psma_pipeline.__version__)

In [ ]:
#@title 1 · data
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from psma_pipeline import load_config, setup_logging
from psma_pipeline.colab import local_overrides, storage_overrides
from psma_pipeline import md_analysis as MA
setup_logging()
cfg = load_config(overrides=(storage_overrides(True) if IN_COLAB else local_overrides(DATA_DIR)) or None)
MD = cfg.outputs_dir / "md_run"
OUT7 = cfg.outputs_dir / "md_analysis"
OUT7.mkdir(parents=True, exist_ok=True)
files = MA.segment_files(MD, "prod")
print(f"finished production segment files: {len(files)}; last starts at {files[-1][0] / 1000:.2f} ns" if files else "no production yet")
STRIDE = 1   #@param {type:"integer"}
# every frame is 10 ps; STRIDE = 5 uses one frame per 50 ps (faster for long runs)

## A · RMSD by region

In [ ]:
ca = MA.ca_table(MD / "solute.pdb")
df, rf = MA.stability(MD, ca, frame_ps=float(cfg.get("md.production.frame_ps", 10)), stride=STRIDE)
print(f"{len(df)} frames, {df['time (ns)'].iloc[0]:.2f}–{df['time (ns)'].iloc[-1]:.2f} ns of production")
S = MA.summary(df)
display(S)
df.to_csv(OUT7 / "rmsd_by_region.csv", index=False); rf.to_csv(OUT7 / "rmsf_ca.csv", index=False); S.to_csv(OUT7 / "rmsd_summary.csv", index=False)

In [ ]:
t = df["time (ns)"]
fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
for c, lab in (("all Cα, global fit", "all Cα (as logged)"), ("ectodomain, crystal (62–750), both chains", "ectodomain, both chains"),
               ("ectodomain, chain A (own fit)", "ectodomain A (own fit)"), ("ectodomain, chain B (own fit)", "ectodomain B (own fit)")):
    ax[0].plot(t, df[c], label=lab)
ax[0].set_title("Whole protein vs crystal ectodomain"); ax[0].legend(fontsize=8)
for ch, ls in (("A", "-"), ("B", "--")):
    for reg in ("cytoplasmic tail (1–19)", "transmembrane helix (20–43)", "stalk (44–61)"):
        ax[1].plot(t, df[f"{reg}, chain {ch}"], ls, label=f"{reg.split(' (')[0]} {ch}")
ax[1].set_title("AlphaFold anchor, relative to the ectodomain"); ax[1].legend(fontsize=7, ncol=2)
for ch in ("A", "B"):
    ax[2].plot(t, df[f"entrance lid (541–548), chain {ch}"], label=f"lid {ch}")
ax[2].set_title("Entrance lid (541–548), relative to the ectodomain"); ax[2].legend(fontsize=8)
for a in ax: a.set_xlabel("production time (ns)"); a.set_ylabel("RMSD (Å)")
plt.tight_layout(); plt.savefig(OUT7 / "rmsd_by_region.png", dpi=150); plt.show()

## B · Flexibility per residue (RMSF)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(18, 6), sharex=True)
for i, ch in enumerate("AB"):
    r = rf[rf["chain"] == ch]
    ax[i].plot(r["uniprot"], r["RMSF (Å)"], lw=0.8)
    ax[i].axvspan(1, 61, color="orange", alpha=0.15, label="AlphaFold anchor (1–61)")
    ax[i].axvspan(541, 548, color="red", alpha=0.2, label="entrance lid")
    ax[i].set_ylabel(f"chain {ch}\nRMSF (Å)")
ax[0].legend(fontsize=8); ax[1].set_xlabel("UniProt residue")
plt.tight_layout(); plt.savefig(OUT7 / "rmsf_ca.png", dpi=150); plt.show()
top = rf[rf["uniprot"] >= 62].sort_values("RMSF (Å)", ascending=False).head(15)
print("Most flexible residues of the crystal ectodomain:"); display(top)

## C–E · Orientation, entrance lid and zinc site

One pass over the trajectory (a few minutes for 200 ns at `STRIDE = 1`). The membrane lies in the xy plane of the box, so its normal is the z axis; the orientation uses the coordinates as written.

The lid is compared with the two crystal structures that calibrate the M2 lid classifier: **3BHX (closed)** and **2XEG (open)**. They are downloaded once into the pipeline cache. If the download fails, the analysis runs without them (helicity and distances only).

In [ ]:
at = MA.atom_table(MD / "solute.pdb")
refs = {}
for pdb_id, state in cfg.lid.get("known_states", {}).items():
    try:
        refs[pdb_id] = (state, MA.reference_lid_ca(cfg, pdb_id))
        print(f"reference {pdb_id} ({state}): {len(refs[pdb_id][1])} Cα")
    except Exception as ex:  # noqa: BLE001
        print(f"reference {pdb_id} not used: {ex}")
res = MA.site_and_shape(MD, at, cfg.lid, refs, frame_ps=float(cfg.get("md.production.frame_ps", 10)), stride=STRIDE)
for k, v in res.items():
    v.to_csv(OUT7 / f"{k.replace(' ', '_')}.csv", index=False)
print(f"{len(res['orientation'])} frames")

### C · Orientation in the membrane

- **TM helix tilt:** angle between each transmembrane helix (Cα 22–42) and the membrane normal. Single-pass helices typically sit at 10–30°.
- **Angle between TM helices:** about 0–40° for a packed pair. A large or growing angle means the helices are splaying apart.
- **Ectodomain tilt:** angle between the membrane normal and the line from the TM pair to the centre of the ectodomain dimer. 0° = standing upright. **Height** is how far the ectodomain centre sits above the TM pair. If the height drops, the ectodomain is leaning over towards the membrane.

In [ ]:
o = res["orientation"]
O = MA.describe(o); display(O); O.to_csv(OUT7 / "orientation_summary.csv", index=False)
t = o["time (ns)"]
fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
for c in ("TM helix tilt, chain A (°)", "TM helix tilt, chain B (°)", "angle between TM helices (°)"):
    ax[0].plot(t, o[c], label=c.replace(" (°)", ""))
ax[0].set_ylabel("angle (°)"); ax[0].set_title("Transmembrane helices"); ax[0].legend(fontsize=8)
ax[1].plot(t, o["ectodomain tilt from the membrane normal (°)"]); ax[1].set_ylabel("tilt (°)"); ax[1].set_title("Ectodomain tilt")
ax[2].plot(t, o["ectodomain height above the TM pair (Å)"], label="height above TM pair")
ax[2].plot(t, o["TM helix separation (Å)"], label="TM helix separation"); ax[2].set_ylabel("Å"); ax[2].legend(fontsize=8)
ax[2].set_title("Distances")
for a in ax: a.set_xlabel("production time (ns)")
plt.tight_layout(); plt.savefig(OUT7 / "orientation.png", dpi=150); plt.show()

### D · Entrance lid (541–548)

The lid is a **helix when closed** and has **no regular structure when open** (this exposes the arene-binding site at Trp541, Arg463 and Arg511). Measures per chain:

- **Helicity:** the share of Cα(i)–Cα(i+3) distances under 6 Å, the same measure and threshold (≥ 0.6 = closed) as the M2 classifier.
- **Lid RMSD to 3BHX (closed) and 2XEG (open):** measured after superposing the chain's ectodomain on each crystal structure. The nearer one is the "nearest reference" state.
- **Lid → zinc and Trp541 → zinc distances:** these grow if the lid swings away from the active site.

9S7U, the starting structure, has a closed lid.

In [ ]:
l = res["lid"]
display(MA.lid_state_summary(l)); L = MA.describe(l); display(L); L.to_csv(OUT7 / "lid_summary.csv", index=False)
t = l["time (ns)"]
fig, ax = plt.subplots(1, 3, figsize=(18, 4.2))
for ch, ls in (("A", "-"), ("B", "--")):
    ax[0].plot(t, l[f"lid helicity, chain {ch}"].rolling(10, min_periods=1).mean(), ls, label=f"chain {ch}")
    for c in [c for c in l.columns if c.startswith("lid RMSD to ") and c.endswith(f"chain {ch} (Å)")]:
        ax[1].plot(t, l[c], ls, label=c.replace("lid RMSD to ", "").replace(" (Å)", ""))
    for c in (f"lid to zinc pair, chain {ch} (Å)", f"Trp541 ring to zinc pair, chain {ch} (Å)"):
        if c in l: ax[2].plot(t, l[c], ls, label=c.replace(" (Å)", ""))
ax[0].axhline(float(cfg.lid.get("classifier", {}).get("helix_fraction_min", 0.6)), color="grey", lw=0.8, ls=":")
ax[0].set_ylim(-0.05, 1.05); ax[0].set_ylabel("helicity (100 ps running mean)"); ax[0].set_title("Lid helicity (above dotted line = closed)")
ax[1].set_ylabel("RMSD (Å)"); ax[1].set_title("Lid vs closed / open crystal structures")
ax[2].set_ylabel("distance (Å)"); ax[2].set_title("Lid and Trp541 to the zinc pair")
for a in ax: a.set_xlabel("production time (ns)"); a.legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT7 / "entrance_lid.png", dpi=150); plt.show()

### E · Binuclear zinc site

The ligands are all N/O atoms within 2.8 Å of each Zn in the first production frame (with the bonded MCPB.py model these are His377, Asp387, Glu425, Asp453, His553 and the bridging hydroxide). If a distance sits more than 0.5 Å above its start for many frames, that ligand is drifting away.

In [ ]:
Z = MA.zinc_summary(res["zinc"], res["zinc ligands"]); display(Z); Z.to_csv(OUT7 / "zinc_summary.csv", index=False)
z = res["zinc"]; t = z["time (ns)"]
fig, ax = plt.subplots(1, 2, figsize=(18, 4.2), sharey=True)
for i, ch in enumerate("AB"):
    for c in [c for c in z.columns if c.startswith(f"{ch}:Zn")]:
        ax[i].plot(t, z[c], lw=0.7, label=c)
    ax[i].set_title(f"chain {ch}"); ax[i].set_xlabel("production time (ns)"); ax[i].legend(fontsize=6, ncol=2)
ax[0].set_ylabel("distance (Å)")
plt.tight_layout(); plt.savefig(OUT7 / "zinc_site.png", dpi=150); plt.show()

## How to read this

- **The whole-protein RMSD is high, but the ectodomain is low (about 1–2.5 Å) and flat:** the drift comes from the AlphaFold anchor (tails, transmembrane helices, stalk). That is expected, and the crystal part is stable.
- **The ectodomain RMSD climbs steadily:** the extracellular part itself is changing. Look at the two chains (own fit) and at the RMSF peaks to see where.
- **The transmembrane helix RMSD is large and steady:** the helices have tilted or moved as a body in the membrane relative to the ectodomain. That says something about how the AlphaFold anchor sits; it is not a sign of instability.
- **The lid RMSD changes:** the entrance lid is moving, which is relevant for ligand access (section D).
- **Orientation (C):** steady tilt angles mean the anchor has settled. A tilt that keeps changing means the membrane anchor is still relaxing, so use the later part of the run for averages.
- **Lid (D):** helicity staying above 0.6 and nearer 3BHX means closed throughout. A drop in helicity and a switch to "nearer 2XEG" is a lid-opening event, worth looking at frame by frame.
- **Zinc (E):** all distances flat with SD ≈ 0.05–0.1 Å means the site is intact. A distance climbing by > 0.5 Å means a ligand is leaving.

Outputs: `outputs/md_analysis/` (part 1: `rmsd_by_region.csv`, `rmsd_summary.csv`, `rmsf_ca.csv`; part 2: `orientation*.csv`, `lid*.csv`, `zinc*.csv`, `zinc_ligands.csv`; and the figures).